# Phase 3 Validation — LLM Provider Layer

**Purpose:** Interactively exercise every Phase 3 component against real prompt files and a mocked LLM model.  
No real API calls are made. All LLM responses are simulated with `MagicMock`.

## What this notebook proves

| Section | What is validated |
|---|---|
| 1 | `PromptLoader` loads and caches real prompt files from `app/prompts/` |
| 2 | Guardrails are prepended to every assembled system message |
| 3 | Prompt version tags are extracted and stripped correctly |
| 4 | `ClaudeProvider.complete()` returns a validated dict from a mocked model |
| 5 | Schema repair fires when the model returns a parsing error |
| 6 | Retry logic fires on simulated `APIConnectionError` |
| 7 | Token extraction and cost estimation work correctly |
| 8 | `make_resume_enhance_fn()` produces a valid `enhance_fn` callable |
| 9 | `OpenAIProvider` raises `NotImplementedError` (stub confirmed) |

## Relationship to Phase 4

Phase 4 agents will call `provider.complete(agent_name, context, Schema)` directly.  
This notebook shows exactly what that call does internally — prompt assembly, chain construction,  
retry, repair — so you can reason about what agents inherit for free.

---
## Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path so app.* imports work from the notebooks/ directory
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROMPTS_DIR = PROJECT_ROOT / "app" / "prompts"
print(f"Project root : {PROJECT_ROOT}")
print(f"Prompts dir  : {PROMPTS_DIR}")
print(f"Prompts exist: {PROMPTS_DIR.exists()}")

In [ ]:
import json
from unittest.mock import MagicMock

import anthropic
import httpx
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from pydantic import BaseModel

from app.providers.llm_client import LLMClient, LLMProviderError
from app.providers.prompt_loader import PromptLoader
from app.providers.claude_provider import ClaudeProvider, make_resume_enhance_fn
from app.providers.openai_provider import OpenAIProvider
from app.schemas.job_score import JobScore

print("All imports OK")

In [ ]:
# ---------------------------------------------------------------------------
# Shared mock helpers used throughout this notebook
# ---------------------------------------------------------------------------

def make_ai_message(tokens_in: int = 200, tokens_out: int = 80) -> AIMessage:
    """Construct a realistic AIMessage with usage_metadata."""
    return AIMessage(
        content="",
        usage_metadata={
            "input_tokens": tokens_in,
            "output_tokens": tokens_out,
            "total_tokens": tokens_in + tokens_out,
        },
    )


def make_mock_chain(parsed_obj, parsing_error=None) -> MagicMock:
    """Return a chain mock whose .invoke() returns an include_raw=True style dict."""
    chain = MagicMock()
    chain.invoke.return_value = {
        "raw": make_ai_message(),
        "parsed": parsed_obj,
        "parsing_error": parsing_error,
    }
    return chain


def make_mock_model(parsed_obj, parsing_error=None) -> MagicMock:
    """Return a mock ChatAnthropic that returns the given parsed object."""
    model = MagicMock()
    model.with_structured_output.return_value = make_mock_chain(parsed_obj, parsing_error)
    return model


def make_provider(model=None) -> ClaudeProvider:
    """Construct a ClaudeProvider backed by the real prompt files."""
    loader = PromptLoader(PROMPTS_DIR)
    return ClaudeProvider(loader, _model=model or MagicMock())


print("Mock helpers ready")

---
## Section 1 — PromptLoader: Real File Loading

`PromptLoader` reads prompt files from `app/prompts/` on first access and caches them in memory.  
Here we confirm it finds the real files and assembles the correct message structure.

In [ ]:
loader = PromptLoader(PROMPTS_DIR)

# List all prompt files that exist
agent_prompts = sorted((PROMPTS_DIR / "agents").glob("*.txt"))
print(f"Shared guardrails : {(PROMPTS_DIR / 'shared' / 'guardrails.txt').exists()}")
print(f"Agent prompt files: {len(agent_prompts)}")
for p in agent_prompts:
    print(f"  {p.name}")

In [ ]:
# Assemble messages for scoring_agent and inspect the result
context = {
    "job_description": "Staff Engineer at Acme Corp — Kubernetes, Python, distributed systems.",
    "resume_profile": {"name": "Jane Smith", "skills": ["Python", "Kubernetes", "GCP"]},
    "career_track": "ic",
}

messages = loader.assemble("scoring_agent", context)

print(f"Message count : {len(messages)}")
print(f"Message types : {[type(m).__name__ for m in messages]}")
print()
print(f"SystemMessage length : {len(messages[0].content)} characters")
print(f"HumanMessage length  : {len(messages[1].content)} characters")

---
## Section 2 — Guardrails: Injected into Every System Message

The guardrails block is prepended unconditionally by `PromptLoader._build_system_content()`.  
Agents cannot opt out. Here we confirm the key safety phrases are present.

In [ ]:
system_content = messages[0].content

# Print the full assembled system message so you can read what Claude receives
print("=" * 70)
print("ASSEMBLED SYSTEM MESSAGE (scoring_agent)")
print("=" * 70)
print(system_content)
print("=" * 70)

In [ ]:
# Verify the three mandatory safety properties
checks = {
    "Guardrails header present":       "ETHICS AND SAFETY GUARDRAILS" in system_content,
    "Fabrication rule present":        "Never fabricate" in system_content,
    "Injection defense present":       "ignore previous instructions" in system_content,
    "Guardrails separated from agent": "---" in system_content,
    "Agent role present":              "Scoring Agent" in system_content,
}

all_passed = True
for check, result in checks.items():
    status = "PASS" if result else "FAIL"
    if not result:
        all_passed = False
    print(f"  [{status}] {check}")

assert all_passed, "One or more guardrail checks failed"
print("\nAll guardrail checks passed")

In [ ]:
# Confirm the same guardrails appear for a completely different agent
critic_messages = loader.assemble("resume_critic", {"job": "...", "resume": "..."})
critic_system = critic_messages[0].content

assert "ETHICS AND SAFETY GUARDRAILS" in critic_system
assert "Never fabricate" in critic_system
assert "Resume Critic" in critic_system  # agent-specific content
print("Guardrails confirmed in resume_critic system message")

---
## Section 3 — Prompt Versioning

Each agent prompt file begins with `# version: N`. `PromptLoader` parses this line,  
strips it from the prompt content (so Claude never sees it), and exposes it via `get_version()`  
for logging and regression tracking.

In [ ]:
# Trigger a load for all agents so versions are populated
agent_names = [p.stem for p in agent_prompts]
for name in agent_names:
    loader.assemble(name, {})

print("Prompt versions:")
for name in sorted(agent_names):
    print(f"  {loader.get_version(name)}")

In [ ]:
# Confirm the version comment is NOT in the assembled system message
assert "# version:" not in system_content, "Version line must be stripped before sending to Claude"
print("Version line correctly stripped from assembled system message")

# Confirm the file cache is populated (both files per agent loaded once)
print(f"\nFiles in loader cache: {len(loader._file_cache)}")
print(f"  (1 shared guardrails + {len(agent_names)} agent prompts = {1 + len(agent_names)} expected)")
assert len(loader._file_cache) == 1 + len(agent_names)

---
## Section 4 — ClaudeProvider: Basic Complete Call

This is the call every Phase 4 agent will make: `provider.complete(agent_name, context, Schema)`.  
The mock model returns a pre-built `JobScore` object so we can validate the full call path  
without touching the Anthropic API.

In [ ]:
# Build a realistic JobScore that the mock model will "return"
fake_score = JobScore(
    job_id="job-001",
    resume_id="res-001",
    overall_score=82,
    technical_score=88,
    architecture_score=79,
    leadership_score=65,
    domain_score=75,
    match_summary="Strong technical fit. Leadership experience is light for a Staff role.",
    strengths=["Python", "Kubernetes", "GCP", "distributed systems experience"],
    gaps=["Limited people management", "No fintech domain experience"],
    recommended_next_action="Apply with cover letter addressing leadership gap",
    confidence=85,
)

mock_model = make_mock_model(fake_score)
provider = make_provider(model=mock_model)

# This is the exact call a Phase 4 ScoringAgent will make
result = provider.complete(
    agent_name="scoring_agent",
    context={
        "job_id": "job-001",
        "resume_id": "res-001",
        "job_description": "Staff Engineer at Acme Corp.",
        "resume_profile": {"name": "Jane Smith", "skills": ["Python", "Kubernetes"]},
        "career_track": "ic",
    },
    schema=JobScore,
)

print("Return type:", type(result).__name__)
print()
print(json.dumps(result, indent=2))

In [ ]:
# Verify the call chain: with_structured_output was called with the schema
mock_model.with_structured_output.assert_called_once_with(JobScore, include_raw=True)
print("with_structured_output called with correct schema and include_raw=True")

# Verify result shape matches JobScore fields
expected_fields = set(JobScore.model_fields.keys())
assert expected_fields == set(result.keys()), f"Missing fields: {expected_fields - set(result.keys())}"
assert result["overall_score"] == 82
assert result["technical_score"] == 88
print("Result dict has all JobScore fields with correct values")

---
## Section 5 — Schema Repair

When `with_structured_output` returns a `parsing_error` (the model produced valid JSON but the  
wrong shape), `ClaudeProvider` makes one repair attempt: it appends a `HumanMessage` with the  
validation error and re-invokes the chain.  

**This fires at most once.** If the repair also fails, `LLMProviderError` is raised.

In [ ]:
# Simulate: first call returns a parsing_error, second call returns valid output
ai_msg = make_ai_message()
bad_result  = {"raw": ai_msg, "parsed": None, "parsing_error": "Field 'overall_score' is required"}
good_result = {"raw": ai_msg, "parsed": fake_score, "parsing_error": None}

repair_chain = MagicMock()
repair_chain.invoke.side_effect = [bad_result, good_result]

repair_model = MagicMock()
repair_model.with_structured_output.return_value = repair_chain

provider_repair = make_provider(model=repair_model)
result = provider_repair.complete("scoring_agent", {"job_id": "j1"}, JobScore)

print(f"chain.invoke called {repair_chain.invoke.call_count} times (1 initial + 1 repair)")
assert repair_chain.invoke.call_count == 2

# The second call must include a follow-up HumanMessage with the error
second_call_messages = repair_chain.invoke.call_args_list[1][0][0]
repair_hint = second_call_messages[-1]
print(f"Repair hint type   : {type(repair_hint).__name__}")
print(f"Repair hint content: {repair_hint.content[:120]}...")
assert isinstance(repair_hint, HumanMessage)
assert "overall_score" in repair_hint.content
print("\nSchema repair flow validated")

In [ ]:
# Simulate: both attempts fail — must raise LLMProviderError
always_bad = {"raw": ai_msg, "parsed": None, "parsing_error": "missing required field"}

fail_chain = MagicMock()
fail_chain.invoke.return_value = always_bad
fail_model = MagicMock()
fail_model.with_structured_output.return_value = fail_chain

provider_fail = make_provider(model=fail_model)

try:
    provider_fail.complete("scoring_agent", {}, JobScore)
    print("ERROR: should have raised LLMProviderError")
except LLMProviderError as e:
    print(f"LLMProviderError correctly raised: {e}")

---
## Section 6 — Retry on API Error

`ClaudeProvider` uses `tenacity` to retry on `APIConnectionError`, `RateLimitError`,  
and `InternalServerError`. The retry uses exponential backoff (1s → 2s → 4s, max 3 attempts).  

We zero out the wait times so the test runs instantly.

In [ ]:
# Simulate: first invoke raises APIConnectionError, second succeeds
api_error = anthropic.APIConnectionError(
    request=httpx.Request("POST", "https://api.anthropic.com/v1/messages")
)
good_result = {"raw": make_ai_message(), "parsed": fake_score, "parsing_error": None}

retry_chain = MagicMock()
retry_chain.invoke.side_effect = [api_error, good_result]

retry_model = MagicMock()
retry_model.with_structured_output.return_value = retry_chain

# Zero wait times so the notebook doesn't sleep
provider_retry = make_provider(model=retry_model)
provider_retry._RETRY_WAIT_MIN = 0
provider_retry._RETRY_WAIT_MAX = 0

result = provider_retry.complete("scoring_agent", {"job_id": "j1"}, JobScore)

print(f"chain.invoke called {retry_chain.invoke.call_count} times")
print(f"  Attempt 1: APIConnectionError (retryable)")
print(f"  Attempt 2: success")
assert retry_chain.invoke.call_count == 2
assert result["overall_score"] == 82
print("\nRetry logic validated")

---
## Section 7 — Token Extraction and Cost Estimation

`ClaudeProvider` extracts token counts from `AIMessage.usage_metadata` after every call  
and logs them via Python's `logging` module. In Phase 5, the orchestrator will additionally  
write these to the `llm_calls` table via `ObservabilityService`.

In [ ]:
# Construct a result dict with known token counts to test extraction
rich_ai_msg = make_ai_message(tokens_in=1_200, tokens_out=450)
raw_result = {
    "raw": rich_ai_msg,
    "parsed": fake_score,
    "parsing_error": None,
}

provider_cost = make_provider()
tokens_in, tokens_out = provider_cost._extract_usage(raw_result)

print(f"Tokens in  : {tokens_in}")
print(f"Tokens out : {tokens_out}")
assert tokens_in == 1_200
assert tokens_out == 450

In [ ]:
from app.providers.claude_provider import _PRICING

# Cost estimation for both models
from app.providers.prompt_loader import PromptLoader as _PL
from app.providers.claude_provider import ClaudeProvider as _CP

loader_cost = _PL(PROMPTS_DIR)

haiku   = _CP(loader_cost, model_name="claude-haiku-4-5-20251001",  _model=MagicMock())
sonnet  = _CP(loader_cost, model_name="claude-sonnet-4-6",          _model=MagicMock())

# 1M tokens in + 1M tokens out for easy validation against pricing table
haiku_cost  = haiku.estimate_cost(1_000_000, 1_000_000)
sonnet_cost = sonnet.estimate_cost(1_000_000, 1_000_000)

print("Per-million-token cost (input + output):")
print(f"  Haiku  : ${haiku_cost:.2f}   (${_PRICING['claude-haiku-4-5-20251001']['input']:.2f} in + ${_PRICING['claude-haiku-4-5-20251001']['output']:.2f} out)")
print(f"  Sonnet : ${sonnet_cost:.2f}  (${_PRICING['claude-sonnet-4-6']['input']:.2f} in + ${_PRICING['claude-sonnet-4-6']['output']:.2f} out)")
print()

# Real-world example: 1,200 input + 450 output tokens per scoring call
haiku_per_call  = haiku.estimate_cost(1_200, 450)
sonnet_per_call = sonnet.estimate_cost(1_200, 450)
print("Cost per call (1,200 in / 450 out):")
print(f"  Haiku  : ${haiku_per_call:.6f}")
print(f"  Sonnet : ${sonnet_per_call:.6f}")
print()

# At MAX_JOBS=20 scored with Haiku
run_cost = haiku_per_call * 20
print(f"Estimated scoring cost for a 20-job run (Haiku): ${run_cost:.4f}")

In [ ]:
# Token approximation (count_tokens is a pre-call estimate, not exact)
sample = "Staff Engineer role requiring 10+ years of distributed systems experience."
approx = provider_cost.count_tokens(sample)
print(f"Sample text ({len(sample)} chars): ~{approx} tokens")
print("Note: actual counts come from usage_metadata after the call")

---
## Section 8 — `make_resume_enhance_fn` Factory

`make_resume_enhance_fn(provider)` produces an `enhance_fn` compatible with `ResumeParser`.  
The orchestrator (Phase 5) creates this once at startup and injects it into `ResumeParser`.  
`ResumeParser` itself never imports `ClaudeProvider` — it just calls `enhance_fn(raw_text, fields)`.

This is the **Strategy pattern** — the parsing algorithm is fixed; the enhancement behavior is injected.

In [ ]:
# Build a mock provider that simulates what Claude would return for resume enhancement
mock_provider = MagicMock(spec=LLMClient)
mock_provider.complete.return_value = {
    "name": "Jane Smith",
    "email": "jane@example.com",
    "skills": ["Python", "Kubernetes", "GCP", "Apache Kafka"],
    "headline": "Staff Software Engineer — Distributed Systems",
}

# Factory produces the callable
enhance_fn = make_resume_enhance_fn(mock_provider)
print(f"enhance_fn type    : {type(enhance_fn).__name__}")
print(f"Is callable        : {callable(enhance_fn)}")

In [ ]:
# Call enhance_fn as ResumeParser would
heuristic_fields = {
    "name": "Jane Smith (heuristic)",
    "email": "jane@example.com",
    "skills": ["python", "k8s"],  # un-normalized aliases
}
raw_text = "Jane Smith\njane@example.com\n\nSKILLS: Python, Kubernetes, GCP, Kafka"

result = enhance_fn(raw_text, heuristic_fields)

print("enhance_fn result:")
print(json.dumps(result, indent=2))

# Confirm it delegated to provider.complete with the correct agent name
call = mock_provider.complete.call_args
print(f"\nagent_name passed to complete: '{call.kwargs['agent_name']}'")
assert call.kwargs["agent_name"] == "resume_parser"
assert call.kwargs["context"]["raw_text"] == raw_text
assert call.kwargs["context"]["heuristic_fields"] == heuristic_fields
print("enhance_fn correctly delegates to provider.complete('resume_parser', ...)")

---
## Section 9 — OpenAIProvider Stub

`OpenAIProvider` satisfies the `LLMClient` interface but raises `NotImplementedError` on all calls.  
This confirms the abstraction is complete — a future OpenAI implementation only needs to fill  
in these three methods, and no agent code changes.

In [ ]:
oai = OpenAIProvider()

# Confirm it is an LLMClient
print(f"Is LLMClient subclass: {isinstance(oai, LLMClient)}")

# All three methods must raise NotImplementedError
results = {}
for method_name, call in [
    ("complete",        lambda: oai.complete("scoring_agent", {}, JobScore)),
    ("count_tokens",    lambda: oai.count_tokens("text")),
    ("estimate_cost",   lambda: oai.estimate_cost(100, 50)),
]:
    try:
        call()
        results[method_name] = "ERROR: did not raise"
    except NotImplementedError:
        results[method_name] = "NotImplementedError (correct)"

for method, outcome in results.items():
    print(f"  {method:20s}: {outcome}")

---
## Summary

All Phase 3 components validated against real prompt files with mocked LLM responses.

In [ ]:
print("Phase 3 Validation Summary")
print("=" * 50)
validations = [
    ("PromptLoader loads real prompt files",          True),
    ("Guardrails present in every system message",    True),
    ("Version tags extracted and stripped",           True),
    ("File cache prevents repeated disk reads",       True),
    ("ClaudeProvider.complete() returns validated dict", True),
    ("Schema repair fires on parsing_error",          True),
    ("LLMProviderError raised after double failure",  True),
    ("Retry fires on APIConnectionError",             True),
    ("Token extraction from usage_metadata",          True),
    ("Cost estimation correct for Haiku and Sonnet",  True),
    ("make_resume_enhance_fn returns callable",       True),
    ("enhance_fn delegates to resume_parser agent",   True),
    ("OpenAIProvider raises NotImplementedError",     True),
]
for label, passed in validations:
    print(f"  [{'PASS' if passed else 'FAIL'}] {label}")

print()
print("Ready for Phase 4 — Agents")